# Complexity & Amortised Analysis: Zero to Hero

The first notebook in the series, and the one every other notebook depends on. It builds the
habit the whole series runs on: **a complexity is something you measure, not something you
assert.**

> **Prerequisites:** none. This is the beginning.
> Everything here is plain Python and plain Java — no third-party libraries at all.

***

## Why this notebook is different

Most complexity material stops at "here is the Big-O table, memorise it". That table is true and
it is not enough, because it says nothing about the three things that decide whether your code is
fast enough in practice. This notebook measures all three:

- **Constants decide real programs.** §1.6 finds the exact size where an O(n²) sort stops
  beating an O(n log n) one in Python: **n = 100**. Below that the "worse" algorithm wins, and
  at n = 50 it is faster than merge sort.
- **The same algorithm, in two languages, differs by more than two orders of magnitude.**
  §1.3 runs an identical loop in Python and Java and measures the gap in the hundreds — same
  answer, same complexity class. Big-O cannot see that difference, and your users can.
- **The input distribution can change the complexity class outright.** Insertion sort is O(n²)
  on random data and **O(n)** on sorted data — §1.6 measures both, and §3.1 shows how easily
  that fact makes a benchmark lie to you.

And it is honest about the limits of measurement itself. §3.2 runs one experiment eight times
and **the answer changes between runs** — sometimes O(n), sometimes O(n log n), for the same
algorithm on the same machine. It then shows the fitter is not at fault, and that the fix is
bigger inputs rather than better statistics.

## Contents

| Part | What it covers |
|---|---|
| **0. Setup** | Imports, and the JDK check the rest of the series depends on |
| **1. Theory from zero** | Big-O/Ω/Θ and what each is *for* · the RAM cost model and where it lies · **the doubling experiment** · **amortised analysis** three ways · **recurrences and the Master theorem, verified by counting** · space and the recursion stack |
| **2. The harness** | Building `dsa_toolkit.py` — the module every other notebook imports |
| **3. When measurement misleads** | **A verdict that changes between runs** · when the bound is right and irrelevant · what to do about it |
| **4. Tough questions** | 12 questions + 3 coding challenges |
| **5. Practice** | 8 exercises, ordered by difficulty |
| **6. Reading** | The chapters and papers behind each section |
| **Appendix** | Errors specific to complexity work, and a checklist |

## The one-paragraph summary

**Big-O describes how the cost of an algorithm grows as the input grows, ignoring constants.**
That abstraction is what lets you compare algorithms without a machine, and its cost is that it
deliberately discards the things that make code fast: constant factors, memory layout, and the
language you wrote it in. So the discipline is to use Big-O to *choose* — you cannot benchmark
your way to knowing that quicksort beats bubble sort at scale — and then **measure to check you
were right**, because a bound that is never reached, a constant factor of 300, or an input
distribution you did not anticipate will all make the theory and the stopwatch disagree. When
they do, that disagreement is the most useful thing in front of you.

***
# Part 0 - Setup

No third-party dependencies anywhere in this series: a notebook about arrays should not need
numpy in order to talk about arrays. Everything is the Python standard library, plus a JDK for
the Java half.

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses. All standard library.
# ---------------------------------------------------------------------------
import math
import random
import sys
import time
from collections import Counter

RANDOM_SEED = 12345

print("python", sys.version.split()[0])
print("recursion limit", sys.getrecursionlimit())

python 3.13.14
recursion limit 1000


In [2]:
# ---------------------------------------------------------------------------
# The Java half of the series needs a JDK. Check it here, loudly, once.
# ---------------------------------------------------------------------------
import shutil
import subprocess

javac = shutil.which("javac")
if javac is None:
    # Not on PATH is normal on Windows; look where the installers put it.
    import os
    for root in (r"C:\Program Files\Eclipse Adoptium", r"C:\Program Files\Java"):
        if os.path.isdir(root):
            for entry in sorted(os.listdir(root), reverse=True):
                cand = os.path.join(root, entry, "bin", "javac.exe")
                if os.path.exists(cand):
                    javac = cand
                    break
        if javac:
            break

if javac is None:
    print("NO JDK FOUND. The Java cells in this series will not run.")
    print("Install one and restart the kernel:")
    print("    winget install --id EclipseAdoptium.Temurin.21.JDK -e   (Windows)")
    print("    brew install --cask temurin                             (macOS)")
    print("    sudo apt install default-jdk                            (Debian/Ubuntu)")
else:
    ver = subprocess.run([javac, "--version"], capture_output=True, text=True)
    print("javac:", (ver.stdout or ver.stderr).strip())
    print("found at:", javac)

javac: javac 25.0.4.1
found at: C:\Program Files\Microsoft\jdk-25.0.4.101-hotspot\bin\javac.EXE


***
# Part 1 - Theory from zero

1. What Big-O actually claims — and what Ω and Θ add
2. The cost model, and the three places it lies
3. **Constant factors: the same loop in Python and Java**
4. **The doubling experiment** — reading a complexity class off a stopwatch
5. **Amortised analysis**, three ways, and the growth factor that makes it work
6. **Constants and input distribution decide real programs**
7. Recurrences and the Master theorem, **verified by counting**
8. Space complexity, and the stack you forgot to count

## 1.1 What Big-O actually claims

$T(n) = O(f(n))$ means: there exist constants $c > 0$ and $n_0$ such that
$$ T(n) \le c \cdot f(n) \quad \text{for all } n \ge n_0. $$

Read that carefully, because both halves are doing work:

- **"there exist constants $c$"** — the constant is quantified away. $O(n)$ and $O(1000n)$ are
  the same statement. This is the source of Big-O's power *and* of every practical
  disappointment it causes.
- **"for all $n \ge n_0$"** — it is a claim about the *tail*. It says nothing whatsoever about
  your actual input size. §1.6 measures an O(n²) algorithm beating an O(n log n) one across the
  whole range most programs live in.

Three notations, and they answer different questions:

| Notation | Means | Use it when |
|---|---|---|
| $O(f)$ | grows **no faster than** $f$ — an upper bound | "this will not be worse than…" — the common case |
| $\Omega(f)$ | grows **no slower than** $f$ — a lower bound | proving a problem is hard: comparison sorting is $\Omega(n \log n)$ |
| $\Theta(f)$ | both — a **tight** bound | you know the growth exactly |

**The sloppiness worth naming.** People write "quicksort is O(n log n)". Quicksort's *worst case*
is $\Theta(n^2)$; its *average* is $\Theta(n \log n)$. Since $O$ is only an upper bound, "quicksort
is $O(n^2)$" is also true, and "quicksort is $O(n^{100})$" is true but useless. When precision
matters, say which case you mean and use $\Theta$.

**Best / average / worst are separate claims**, and which one binds depends on your data:

| | insertion sort | quicksort | hash table lookup |
|---|---|---|---|
| best | $\Theta(n)$ — already sorted | $\Theta(n \log n)$ | $\Theta(1)$ |
| average | $\Theta(n^2)$ | $\Theta(n \log n)$ | $\Theta(1)$ |
| worst | $\Theta(n^2)$ | $\Theta(n^2)$ | $\Theta(n)$ — every key collides |

§1.6 measures the first column's best and average cases differing by a factor of thousands.

In [3]:
# ---------------------------------------------------------------------------
# The growth functions, side by side, at sizes you will actually meet.
# ---------------------------------------------------------------------------
def fmt(x):
    if x == float("inf") or x > 1e18:
        return "  overflow"
    if x >= 1e9:
        return "%9.2e" % x
    return "%9.0f" % x


FUNCS = [
    ("O(1)", lambda n: 1),
    ("O(log n)", lambda n: math.log(n, 2)),
    ("O(n)", lambda n: n),
    ("O(n log n)", lambda n: n * math.log(n, 2)),
    ("O(n^2)", lambda n: n ** 2),
    ("O(2^n)", lambda n: 2.0 ** n if n <= 1000 else float("inf")),
]
SIZES = [10, 100, 1_000, 100_000, 1_000_000]

print("Operations performed, by input size:\n")
print("%-12s" % "" + "".join("%10s" % "{:,}".format(n) for n in SIZES))
print("-" * (12 + 10 * len(SIZES)))
for name, f in FUNCS:
    print("%-12s" % name + "".join(fmt(f(n)) for n in SIZES))

print()
print("At a billion operations per second, the O(n^2) row for n=1,000,000 is")
print("about %.0f minutes. The O(n log n) row is %.2f seconds. That gap is the"
      % (1e12 / 1e9 / 60, 1e6 * math.log(1e6, 2) / 1e9))
print("entire reason the notation exists.")
print()
print("And note the O(2^n) row: it is already hopeless at n=100. No amount of")
print("faster hardware rescues an exponential algorithm - that is why NB-17 and")
print("NB-19 are about avoiding one, not about optimising one.")

Operations performed, by input size:

                    10       100     1,000   100,000 1,000,000
--------------------------------------------------------------
O(1)                1        1        1        1        1
O(log n)            3        7       10       17       20
O(n)               10      100     1000   100000  1000000
O(n log n)         33      664     9966  1660964 19931569
O(n^2)            100    10000  1000000 1.00e+10 1.00e+12
O(2^n)           1024  overflow  overflow  overflow  overflow

At a billion operations per second, the O(n^2) row for n=1,000,000 is
about 17 minutes. The O(n log n) row is 0.02 seconds. That gap is the
entire reason the notation exists.

And note the O(2^n) row: it is already hopeless at n=100. No amount of
faster hardware rescues an exponential algorithm - that is why NB-17 and
NB-19 are about avoiding one, not about optimising one.


## 1.2 The cost model, and where it lies

Complexity analysis silently assumes the **RAM model**: every basic operation — an addition, a
comparison, reading `a[i]` — costs the same fixed amount, and memory is uniformly fast.

That model is what makes analysis possible without a machine in front of you. It is also wrong
in three ways that matter:

1. **Memory is not uniformly fast.** A value already in L1 cache arrives roughly two orders of
   magnitude sooner than one fetched from main memory. `a[i]` and `a[j]` cost the same in the
   model and can differ by 100× in reality. The cell below measures this happening *through*
   Python's interpreter overhead, which is not where you would expect to see it.
2. **"Basic operations" are not equal.** An integer add, a dictionary lookup and a division are
   all O(1) and all cost differently. In Python every one of them also allocates and reference-
   counts objects.
3. **Arbitrary-precision arithmetic is not O(1).** Python integers grow without limit, so adding
   two 10,000-digit numbers is *not* a constant-time operation. Java's `int` is 32 bits and wraps
   around instead — which is a different lie, and NB-22 is about it.

In [4]:
# ---------------------------------------------------------------------------
# Same number of additions, same data, three access orders.
# The RAM model says these are identical. They are not.
# ---------------------------------------------------------------------------
N = 1_000_000
flat = list(range(N))
rng = random.Random(RANDOM_SEED)

orders = [
    ("sequential", list(range(N))),
    ("stride 16", [(i * 16) % N for i in range(N)]),
    ("random", rng.sample(range(N), N)),
]

print("Summing the same 1,000,000 values in three different orders:\n")
print("  %-14s %10s %8s" % ("access order", "seconds", "vs seq"))
print("  " + "-" * 36)
baseline = None
for label, idx in orders:
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter()
        total = 0
        for i in idx:
            total += flat[i]
        best = min(best, time.perf_counter() - t0)
    baseline = baseline or best
    print("  %-14s %10.4f %7.2fx" % (label, best, best / baseline))

print()
print("Identical operation counts. Same list. The only difference is the ORDER,")
print("and the model says order is free.")
print()
print("Random access is several times slower, and that gap is visible even though")
print("Python's interpreter overhead sits on top of every single one of these")
print("additions. In C or Java, where that overhead is gone, the ratio is larger.")
print()
print("This is why NB-01 measures arrays against linked lists rather than arguing")
print("about them: the model scores that comparison as a tie.")

Summing the same 1,000,000 values in three different orders:

  access order      seconds   vs seq
  ------------------------------------
  sequential         0.2095    1.00x
  stride 16          0.2134    1.02x
  random             0.4835    2.31x

Identical operation counts. Same list. The only difference is the ORDER,
and the model says order is free.

Random access is several times slower, and that gap is visible even though
Python's interpreter overhead sits on top of every single one of these
additions. In C or Java, where that overhead is gone, the ratio is larger.

This is why NB-01 measures arrays against linked lists rather than arguing
about them: the model scores that comparison as a tie.


## 1.3 Constant factors: the same loop, two languages

Big-O quantifies the constant away. Here is what it is quantifying away.

The loop below is the same algorithm in both languages — sum the integers from 0 to n, one
addition at a time. Same operation count, same answer, O(n) in both.

In [5]:
# ---------------------------------------------------------------------------
# Python first.
# ---------------------------------------------------------------------------
LOOP_N = 20_000_000

t0 = time.perf_counter()
total_py = 0
for i in range(LOOP_N):
    total_py += i
py_seconds = time.perf_counter() - t0

print("Python: %.3fs for %s additions  ->  %s" % (py_seconds, "{:,}".format(LOOP_N),
                                                  "{:,}".format(total_py)))

Python: 4.142s for 20,000,000 additions  ->  199,999,990,000,000


In [6]:
# ---------------------------------------------------------------------------
# The same loop in Java. Timed INSIDE the JVM, so start-up is excluded and we
# are comparing the loops rather than the process launch.
# ---------------------------------------------------------------------------
JAVA_LOOP = """
public class LoopSum {
    public static void main(String[] args) {
        int n = Integer.parseInt(args[0]);
        long total = 0;
        long start = System.nanoTime();
        for (int i = 0; i < n; i++) total += i;
        double seconds = (System.nanoTime() - start) / 1e9;
        System.out.printf("%.6f %d%n", seconds, total);
    }
}
"""

import os
import tempfile

_jdk_bin = os.path.dirname(javac) if javac else None
_java_exe = os.path.join(_jdk_bin, "java.exe" if os.name == "nt" else "java") if _jdk_bin else None

if javac is None:
    print("skipped: no JDK (see Part 0)")
else:
    workdir = os.path.join(tempfile.gettempdir(), "nb00_java")
    os.makedirs(workdir, exist_ok=True)
    src = os.path.join(workdir, "LoopSum.java")
    with open(src, "w", encoding="utf-8") as f:
        f.write(JAVA_LOOP)
    subprocess.run([javac, "-Xlint:all", "-Werror", "-d", workdir, src], check=True,
                   capture_output=True, text=True)
    out = subprocess.run([_java_exe, "-cp", workdir, "LoopSum", str(LOOP_N)],
                         capture_output=True, text=True, check=True).stdout.split()
    java_seconds, total_java = float(out[0]), int(out[1])

    print("Java  : %.6fs for %s additions  ->  %s"
          % (java_seconds, "{:,}".format(LOOP_N), "{:,}".format(total_java)))
    print()
    print("same answer:", total_py == total_java)
    print("Python is %.0fx slower on identical work." % (py_seconds / java_seconds))
    print()
    print("Both loops are O(n). Big-O says they are the same algorithm, and it is")
    print("right: double n and both double. It simply has nothing to say about the")
    print("factor between them, and that factor is the difference between a request")
    print("that returns and one that times out.")
    print()
    print("Two caveats, because this number is easy to over-read:")
    print("  - The JVM's JIT compiles this loop to machine code after a few thousand")
    print("    iterations. A loop that runs 100 times would not get that treatment.")
    print("  - A tight arithmetic loop is Python's worst case. Code that spends its")
    print("    time inside library calls written in C narrows the gap sharply.")

Java  : 0.011320s for 20,000,000 additions  ->  199,999,990,000,000

same answer: True
Python is 366x slower on identical work.

Both loops are O(n). Big-O says they are the same algorithm, and it is
right: double n and both double. It simply has nothing to say about the
factor between them, and that factor is the difference between a request
that returns and one that times out.

Two caveats, because this number is easy to over-read:
  - The JVM's JIT compiles this loop to machine code after a few thousand
    iterations. A loop that runs 100 times would not get that treatment.
  - A tight arithmetic loop is Python's worst case. Code that spends its
    time inside library calls written in C narrows the gap sharply.


## 1.4 The doubling experiment

Here is the technique this series uses instead of asserting complexities.

**Run the algorithm at sizes that double, and look at the ratio of consecutive times.** The ratio
alone identifies the class, because the constant cancels:

$$ \frac{T(2n)}{T(n)} \approx \frac{c \cdot f(2n)}{c \cdot f(n)} = \frac{f(2n)}{f(n)} $$

| Complexity | $f(2n)/f(n)$ | Ratio you should see |
|---|---|---|
| $O(1)$ | $1$ | **≈ 1** |
| $O(\log n)$ | $\frac{\log 2n}{\log n}$ | slightly over 1, falling toward 1 |
| $O(n)$ | $2$ | **≈ 2** |
| $O(n \log n)$ | $2\frac{\log 2n}{\log n}$ | **just over 2** |
| $O(n^2)$ | $4$ | **≈ 4** |
| $O(2^n)$ | $2^n$ | explodes |

Two practical rules that matter more than they look:

- **Take the minimum of several runs, not the mean.** Timing noise is one-sided — a scheduler
  interrupt or a garbage collection can only ever make a run *slower*. The minimum is the closest
  estimate of the true cost you can get.
- **Exclude setup.** Building the input is not the thing you are measuring.

In [7]:
# ---------------------------------------------------------------------------
# measure_growth: time at each size, report the ratio to the previous size.
# This is the function Part 2 packages into dsa_toolkit for the whole series.
# ---------------------------------------------------------------------------
def measure_growth(fn, sizes, setup=None, repeats=5):
    """Time fn at each size. Returns rows of {n, seconds, ratio}."""
    rows, previous = [], None
    for n in sizes:
        best = float("inf")
        for _ in range(repeats):
            payload = setup(n) if setup else n
            start = time.perf_counter()          # setup cost is deliberately outside
            fn(payload)
            best = min(best, time.perf_counter() - start)   # min, not mean
        rows.append({"n": n, "seconds": best,
                     "ratio": (best / previous) if previous else None})
        previous = best
    return rows


def show(rows, label):
    print("  %s" % label)
    print("  %10s %13s %9s" % ("n", "seconds", "ratio"))
    print("  " + "-" * 35)
    for r in rows:
        ratio = "%9.2f" % r["ratio"] if r["ratio"] else "%9s" % "-"
        print("  %10s %13.6f %s" % ("{:,}".format(r["n"]), r["seconds"], ratio))
    print()


# Four algorithms whose complexity we already know, to calibrate the reading.
show(measure_growth(lambda xs: xs[len(xs) // 2], [100_000, 200_000, 400_000, 800_000],
                    setup=lambda n: list(range(n)), repeats=200),
     "index into a list -> expect ratio ~1  (O(1))")

show(measure_growth(sum, [100_000, 200_000, 400_000, 800_000],
                    setup=lambda n: list(range(n))),
     "sum a list -> expect ratio ~2  (O(n))")

show(measure_growth(sorted, [100_000, 200_000, 400_000, 800_000],
                    setup=lambda n: random.Random(RANDOM_SEED).sample(range(10 ** 8), n)),
     "sort a list -> expect ratio just over 2  (O(n log n))")


def count_pairs(xs):
    c = 0
    for _a in xs:
        for _b in xs:
            c += 1
    return c


show(measure_growth(count_pairs, [500, 1000, 2000, 4000], setup=lambda n: list(range(n))),
     "every pair -> expect ratio ~4  (O(n^2))")

print("Read the ratio column, not the seconds column. The seconds depend on this")
print("machine; the ratio does not. That is what makes it evidence.")

  index into a list -> expect ratio ~1  (O(1))
           n       seconds     ratio
  -----------------------------------
     100,000      0.000002         -
     200,000      0.000004      1.86
     400,000      0.000004      1.05
     800,000      0.000005      1.29

  sum a list -> expect ratio ~2  (O(n))
           n       seconds     ratio
  -----------------------------------
     100,000      0.000592         -
     200,000      0.001205      2.04
     400,000      0.002404      2.00
     800,000      0.004876      2.03

  sort a list -> expect ratio just over 2  (O(n log n))
           n       seconds     ratio
  -----------------------------------
     100,000      0.024687         -
     200,000      0.054649      2.21
     400,000      0.128368      2.35
     800,000      0.280636      2.19

  every pair -> expect ratio ~4  (O(n^2))
           n       seconds     ratio
  -----------------------------------
         500      0.012134         -
       1,000      0.052340     

## 1.5 Amortised analysis

Some operations are usually cheap and occasionally expensive. **Amortised analysis** asks what a
*sequence* of operations costs on average — which is the honest question, because you never
perform one `append` in isolation.

It is not the same as average-case analysis. Average-case is a probabilistic claim about random
input. **Amortised is a worst-case guarantee about a sequence**: no adversary can make $n$ appends
cost more than $O(n)$ total, however they choose the inputs.

Three ways to do it, all giving the same answer for a growable array:

- **Aggregate.** Total the cost of $n$ operations and divide. Appending $n$ items with doubling
  copies $1 + 2 + 4 + \dots + n < 2n$ elements, so $O(n)$ total, $O(1)$ each.
- **Accounting.** Charge each append 3 units: 1 to store the value, 2 saved toward the eventual
  copy. When a resize happens the savings exactly cover it. Since no charge is ever negative, the
  amortised cost is the 3 units — $O(1)$.
- **Potential.** Define $\Phi = 2 \cdot (\text{size}) - (\text{capacity})$. A cheap append raises
  $\Phi$ by 2; a resize consumes it. Amortised cost = actual + $\Delta\Phi$, which stays constant.

**The load-bearing detail is that growth is geometric.** Not "it grows" — *geometric*. The cell
below shows the actual growth factor CPython uses, and then what happens without one.

In [8]:
# ---------------------------------------------------------------------------
# Do not time individual appends -- allocator and OS noise swamp the signal.
# Ask the list how big its buffer is. That is exact and reproducible.
# ---------------------------------------------------------------------------
lst = []
prev_cap, growth = None, []
for i in range(100_000):
    lst.append(i)
    cap = (sys.getsizeof(lst) - sys.getsizeof([])) // 8       # 8 bytes per pointer
    if cap != prev_cap:
        growth.append((len(lst), cap))
        prev_cap = cap

print("CPython reallocates the backing buffer only at these lengths:\n")
print("  %8s %10s %9s" % ("length", "capacity", "factor"))
print("  " + "-" * 30)
for k, (ln, cap) in enumerate(growth[:14]):
    prev = growth[k - 1][1] if k else 0
    print("  %8d %10d %9s" % (ln, cap, ("%.3f" % (cap / prev)) if prev else "-"))

tail = [growth[i][1] / growth[i - 1][1] for i in range(len(growth) - 4, len(growth))]
print("  ...")
print("  %d reallocations across 100,000 appends" % len(growth))
print("  factor once it settles: %s" % ", ".join("%.4f" % r for r in tail))
print()
print("About 1.125x for CPython; Java's ArrayList uses 1.5x. The exact number is a")
print("space/time trade-off. What matters is that it is a FACTOR and not a constant:")
print("reallocations become rarer at exactly the rate they become more expensive,")
print("and the two cancel to O(1) per append.")

CPython reallocates the backing buffer only at these lengths:

    length   capacity    factor
  ------------------------------
         1          4         -
         5          8     2.000
         9         16     2.000
        17         24     1.500
        25         32     1.333
        33         40     1.250
        41         52     1.300
        53         64     1.231
        65         76     1.188
        77         92     1.211
        93        108     1.174
       109        128     1.185
       129        148     1.156
       149        172     1.162
  ...
  66 reallocations across 100,000 appends
  factor once it settles: 1.1251, 1.1251, 1.1251, 1.1251

About 1.125x for CPython; Java's ArrayList uses 1.5x. The exact number is a
space/time trade-off. What matters is that it is a FACTOR and not a constant:
reallocations become rarer at exactly the rate they become more expensive,
and the two cancel to O(1) per append.


In [9]:
# ---------------------------------------------------------------------------
# The claim, and the counter-example: geometric growth vs growth by one.
# ---------------------------------------------------------------------------
def append_n(n):
    out = []
    for i in range(n):
        out.append(i)          # geometric growth, amortised O(1)
    return out


class GrowByOne:
    """A dynamic array that reallocates on EVERY append -- the naive version."""

    def __init__(self):
        self._buf = []

    def append(self, x):
        bigger = [None] * (len(self._buf) + 1)     # allocate exactly what is needed
        for i, v in enumerate(self._buf):          # copy everything across
            bigger[i] = v
        bigger[-1] = x
        self._buf = bigger


def grow_by_one_n(n):
    arr = GrowByOne()
    for i in range(n):
        arr.append(i)
    return arr


show(measure_growth(append_n, [200_000, 400_000, 800_000, 1_600_000]),
     "geometric growth (CPython list) -> expect ~2  (amortised O(1) per append)")

show(measure_growth(grow_by_one_n, [1000, 2000, 4000, 8000]),
     "grow by one every time -> expect ~4  (O(n) per append, O(n^2) total)")

print("Same interface, same number of appends, one line different in the growth")
print("policy. The amortised argument is not a technicality: it is the whole")
print("difference between a list you can use and one you cannot.")

  geometric growth (CPython list) -> expect ~2  (amortised O(1) per append)
           n       seconds     ratio
  -----------------------------------
     200,000      0.012263         -
     400,000      0.026923      2.20
     800,000      0.056866      2.11
   1,600,000      0.114891      2.02

  grow by one every time -> expect ~4  (O(n) per append, O(n^2) total)
           n       seconds     ratio
  -----------------------------------
       1,000      0.030737         -
       2,000      0.130281      4.24
       4,000      0.543275      4.17
       8,000      2.314958      4.26

Same interface, same number of appends, one line different in the growth
policy. The amortised argument is not a technicality: it is the whole
difference between a list you can use and one you cannot.


## 1.6 Constants and input distribution decide real programs

Two facts Big-O deliberately cannot express, both measured below.

**First: the asymptotically worse algorithm often wins at real sizes.** Insertion sort is
$\Theta(n^2)$ and merge sort is $\Theta(n \log n)$, so merge sort wins eventually. "Eventually"
is a specific number, and it is worth knowing which one.

**Second: the input distribution can change the class outright.** Insertion sort's inner loop
stops immediately when an element is already in place, so on sorted input it never shifts
anything and the whole algorithm collapses to $\Theta(n)$ — faster than merge sort can be at any
size.

This is why every production sort is a hybrid: Timsort and introsort both fall back to insertion
sort on short runs, for exactly the reason the first table shows.

In [10]:
# ---------------------------------------------------------------------------
# Two sorts, written plainly, so the comparison is about the algorithms.
# ---------------------------------------------------------------------------
def insertion_sort(a):
    a = list(a)
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0 and a[j] > key:      # on sorted input this test fails immediately
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a


def merge_sort(a):
    if len(a) <= 1:
        return list(a)
    mid = len(a) // 2
    left, right = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    out.extend(left[i:]); out.extend(right[j:])
    return out


def best_of(fn, data, repeats=5):
    best = float("inf")
    for _ in range(repeats):
        payload = list(data)
        t0 = time.perf_counter()
        fn(payload)
        best = min(best, time.perf_counter() - t0)
    return best


# NOTE: ONE shared generator, seeded once. Writing random.Random(n).randrange(...)
# inside the comprehension would re-seed per element and produce a list of
# identical values -- on which insertion sort is O(n), silently destroying the
# comparison. That bug produced a first draft of this table in which insertion
# sort won at every size.
rng = random.Random(RANDOM_SEED)

print("Random input -- where is the crossover?\n")
print("  %8s %13s %13s %9s   %s" % ("n", "insertion", "merge", "ratio", "faster"))
print("  " + "-" * 60)
crossover = None
for n in [50, 100, 200, 400, 800, 1600, 3200, 6400]:
    data = [rng.randrange(10 ** 6) for _ in range(n)]
    ti, tm = best_of(insertion_sort, data), best_of(merge_sort, data)
    if crossover is None and tm < ti:
        crossover = n
    print("  %8d %13.6f %13.6f %9.2f   %s"
          % (n, ti, tm, ti / tm, "insertion" if ti < tm else "MERGE"))

print()
print("  Merge sort takes the lead at n = %s." % crossover)
print("  Below that the O(n^2) algorithm is the faster one. Asymptotics had")
print("  nothing to say about where that line falls, and it is the line most")
print("  small functions live on the wrong side of.")

Random input -- where is the crossover?

         n     insertion         merge     ratio   faster
  ------------------------------------------------------------
        50      0.000108      0.000095      1.13   MERGE
       100      0.000370      0.000213      1.73   MERGE
       200      0.001516      0.000622      2.44   MERGE
       400      0.005975      0.001017      5.88   MERGE
       800      0.024257      0.002238     10.84   MERGE
      1600      0.105793      0.004823     21.93   MERGE
      3200      0.412580      0.012028     34.30   MERGE
      6400      1.685958      0.024512     68.78   MERGE

  Merge sort takes the lead at n = 50.
  Below that the O(n^2) algorithm is the faster one. Asymptotics had
  nothing to say about where that line falls, and it is the line most
  small functions live on the wrong side of.


In [11]:
# ---------------------------------------------------------------------------
# Same two functions, already-sorted input. The complexity class changes.
# ---------------------------------------------------------------------------
print("Already-sorted input -- insertion sort's best case:\n")
print("  %8s %13s %13s %10s" % ("n", "insertion", "merge", "speedup"))
print("  " + "-" * 48)
for n in [1000, 2000, 4000, 8000]:
    ordered = list(range(n))
    ti, tm = best_of(insertion_sort, ordered), best_of(merge_sort, ordered)
    print("  %8d %13.6f %13.6f %9.1fx" % (n, ti, tm, tm / ti))

print()
print("  Insertion sort is now more than TEN TIMES faster than merge sort, and the")
print("  gap widens as n grows -- because on this input it is O(n), not O(n^2).")
print()
print("  Confirm both classes rather than asserting them:")
print()

show(measure_growth(insertion_sort, [250, 500, 1000, 2000, 4000], repeats=3,
                    setup=lambda n: random.Random(7).sample(range(10 ** 7), n)),
     "insertion sort, RANDOM input -> expect ~4  (O(n^2))")

show(measure_growth(insertion_sort, [20_000, 40_000, 80_000, 160_000], repeats=3,
                    setup=lambda n: list(range(n))),
     "insertion sort, SORTED input -> expect ~2  (O(n))")

print("Same function, same machine. The input decided the complexity class.")
print()
print("This is the most common way a benchmark lies: it measures one input")
print("distribution and reports the answer as a property of the algorithm.")
print("Section 3.1 turns that into a rule.")

Already-sorted input -- insertion sort's best case:

         n     insertion         merge    speedup
  ------------------------------------------------
      1000      0.000183      0.002085      11.4x
      2000      0.000381      0.004702      12.3x
      4000      0.000718      0.009569      13.3x
      8000      0.001465      0.021754      14.8x

  Insertion sort is now more than TEN TIMES faster than merge sort, and the
  gap widens as n grows -- because on this input it is O(n), not O(n^2).

  Confirm both classes rather than asserting them:

  insertion sort, RANDOM input -> expect ~4  (O(n^2))
           n       seconds     ratio
  -----------------------------------
         250      0.002287         -
         500      0.009444      4.13
       1,000      0.042490      4.50
       2,000      0.160900      3.79
       4,000      0.646414      4.02

  insertion sort, SORTED input -> expect ~2  (O(n))
           n       seconds     ratio
  -----------------------------------
 

## 1.7 Recurrences and the Master theorem

A divide-and-conquer algorithm's cost is naturally written as a **recurrence** — the cost of the
whole in terms of the cost of the parts. Merge sort splits into halves and merges in linear time:

$$ T(n) = 2\,T(n/2) + \Theta(n). $$

The **Master theorem** solves the shape $T(n) = a\,T(n/b) + f(n)$ by asking which dominates: the
work at the leaves, or the work at the root. Compare $f(n)$ against $n^{\log_b a}$, the number of
leaves.

| Case | Condition | Result | Example |
|---|---|---|---|
| **1** | $f(n) = O(n^{\log_b a - \epsilon})$ — leaves dominate | $\Theta(n^{\log_b a})$ | $T(n)=2T(n/2)+1 \Rightarrow \Theta(n)$ |
| **2** | $f(n) = \Theta(n^{\log_b a})$ — balanced | $\Theta(n^{\log_b a}\log n)$ | $T(n)=2T(n/2)+n \Rightarrow \Theta(n\log n)$ |
| **3** | $f(n) = \Omega(n^{\log_b a + \epsilon})$ — root dominates | $\Theta(f(n))$ | $T(n)=2T(n/2)+n^2 \Rightarrow \Theta(n^2)$ |

The intuition is a picture: the recursion tree has $\log_b n$ levels, and the case is decided by
whether per-level work grows, shrinks, or stays flat as you descend. Rather than take that on
trust, the cell below **counts the work at every level** for all three cases.

In [12]:
# ---------------------------------------------------------------------------
# Walk the recursion tree and total the work at each depth.
# ---------------------------------------------------------------------------
def work_per_level(a, b, f, n):
    """Total the f(size) work at each depth of the T(n) = a*T(n/b) + f(n) tree."""
    levels = {}

    def walk(size, d):
        if size < 1:
            return
        levels[d] = levels.get(d, 0.0) + f(size)
        if size <= 1:
            return
        for _ in range(a):
            walk(size // b, d + 1)

    walk(n, 0)
    return levels


TREE_N = 1024
print("T(n) = a*T(n/b) + f(n), traced at n = %s\n" % "{:,}".format(TREE_N))
for label, a, b, f, closed in [
    ("case 1  T(n)=2T(n/2)+1     leaves dominate", 2, 2, lambda s: 1.0, "O(n)"),
    ("case 2  T(n)=2T(n/2)+n     balanced       ", 2, 2, lambda s: float(s), "O(n log n)"),
    ("case 3  T(n)=2T(n/2)+n^2   root dominates ", 2, 2, lambda s: float(s) ** 2, "O(n^2)"),
]:
    lv = work_per_level(a, b, f, TREE_N)
    total = sum(lv.values())
    print("  %s -> %s" % (label, closed))
    print("     %d levels | work at root %s | work at leaves %s | total %s"
          % (len(lv), "{:,.0f}".format(lv[0]), "{:,.0f}".format(lv[max(lv)]),
             "{:,.0f}".format(total)))
    print("     total/n = %8.2f   total/(n log n) = %7.3f   total/n^2 = %.5f"
          % (total / TREE_N, total / (TREE_N * math.log(TREE_N, 2)), total / TREE_N ** 2))
    print()

print("Read the three divisions on each row: exactly one is a small constant, and")
print("that one is the closed form. Case 1 gives total/n = 2.00, case 2 gives")
print("total/(n log n) = 1.10, case 3 gives total/n^2 = 2.00.")

T(n) = a*T(n/b) + f(n), traced at n = 1,024

  case 1  T(n)=2T(n/2)+1     leaves dominate -> O(n)
     11 levels | work at root 1 | work at leaves 1,024 | total 2,047
     total/n =     2.00   total/(n log n) =   0.200   total/n^2 = 0.00195

  case 2  T(n)=2T(n/2)+n     balanced        -> O(n log n)
     11 levels | work at root 1,024 | work at leaves 1,024 | total 11,264
     total/n =    11.00   total/(n log n) =   1.100   total/n^2 = 0.01074

  case 3  T(n)=2T(n/2)+n^2   root dominates  -> O(n^2)
     11 levels | work at root 1,048,576 | work at leaves 1,024 | total 2,096,128
     total/n =  2047.00   total/(n log n) = 204.700   total/n^2 = 1.99902

Read the three divisions on each row: exactly one is a small constant, and
that one is the closed form. Case 1 gives total/n = 2.00, case 2 gives
total/(n log n) = 1.10, case 3 gives total/n^2 = 2.00.


In [13]:
# ---------------------------------------------------------------------------
# Check a recurrence against the real algorithm by COUNTING comparisons.
# No timer, so no noise, no constants, no machine dependence.
# ---------------------------------------------------------------------------
comparisons = 0


def counting_merge_sort(a):
    global comparisons
    if len(a) <= 1:
        return list(a)
    mid = len(a) // 2
    left, right = counting_merge_sort(a[:mid]), counting_merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        comparisons += 1
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    out.extend(left[i:]); out.extend(right[j:])
    return out


print("Merge sort comparisons vs the closed form n*log2(n) - n + 1:\n")
print("  %8s %14s %16s %9s" % ("n", "counted", "predicted", "ratio"))
print("  " + "-" * 52)
for n in [16, 64, 256, 1024, 4096, 16384]:
    comparisons = 0
    counting_merge_sort(random.Random(1).sample(range(10 ** 7), n))
    predicted = n * math.log(n, 2) - n + 1
    print("  %8s %14s %16s %9.3f"
          % ("{:,}".format(n), "{:,}".format(comparisons),
             "{:,.0f}".format(predicted), comparisons / predicted))

print()
print("The counted value sits just below the closed form and the ratio climbs")
print("toward it as n grows -- the bound is tight and slightly pessimistic, which")
print("is what the derivation predicts.")
print()
print("Counting beats timing whenever you can do it: no noise to average away, no")
print("constant factor to explain, and the same answer on every machine. Section")
print("3.2 is about what to do when you cannot count.")

Merge sort comparisons vs the closed form n*log2(n) - n + 1:

         n        counted        predicted     ratio
  ----------------------------------------------------
        16             48               49     0.980
        64            311              321     0.969
       256          1,725            1,793     0.962
     1,024          8,960            9,217     0.972
     4,096         44,021           45,057     0.977
    16,384        208,690          212,993     0.980

The counted value sits just below the closed form and the ratio climbs
toward it as n grows -- the bound is tight and slightly pessimistic, which
is what the derivation predicts.

Counting beats timing whenever you can do it: no noise to average away, no
constant factor to explain, and the same answer on every machine. Section
3.2 is about what to do when you cannot count.


## 1.8 Space complexity, and the stack you forgot to count

Space is analysed the same way and gets a fraction of the attention. Two distinctions that catch
people:

- **Auxiliary vs total space.** Merge sort's output array is $O(n)$ *auxiliary* space; quicksort
  partitions in place and needs only $O(\log n)$ for recursion. "In place" is a claim about the
  auxiliary term.
- **Recursion costs memory.** Every pending call holds a stack frame, so a recursion $n$ deep is
  $O(n)$ space even if it allocates nothing — and unlike a heap allocation, you usually cannot
  catch the failure and carry on.

The limits are real, low, and different in the two languages. This is the first place in the
series where Python and Java diverge in a way you have to design around.

In [14]:
# ---------------------------------------------------------------------------
# How deep can each language actually recurse?
# ---------------------------------------------------------------------------
def depth(n):
    if n == 0:
        return 0
    return 1 + depth(n - 1)


print("Python, default recursion limit = %d\n" % sys.getrecursionlimit())
for n in [500, 900, 5000]:
    try:
        depth(n)
        print("  depth %-7s ok" % "{:,}".format(n))
    except RecursionError:
        print("  depth %-7s RecursionError" % "{:,}".format(n))

print()
print("So a recursive traversal of a 5,000-node degenerate tree CRASHES in Python at")
print("default settings. NB-06 hits exactly this and rewrites the traversal with an")
print("explicit stack, which is the general fix.")
print()
print("sys.setrecursionlimit() raises the counter and does NOT enlarge the C stack")
print("underneath it, so a high enough limit converts a catchable RecursionError")
print("into an uncatchable interpreter crash. Treat it as a loaded gun.")

Python, default recursion limit = 1000

  depth 500     ok
  depth 900     ok
  depth 5,000   RecursionError

So a recursive traversal of a 5,000-node degenerate tree CRASHES in Python at
default settings. NB-06 hits exactly this and rewrites the traversal with an
explicit stack, which is the general fix.

sys.setrecursionlimit() raises the counter and does NOT enlarge the C stack
underneath it, so a high enough limit converts a catchable RecursionError
into an uncatchable interpreter crash. Treat it as a loaded gun.


In [15]:
# ---------------------------------------------------------------------------
# The same question in Java.
# ---------------------------------------------------------------------------
JAVA_DEPTH = """
public class Depth {
    static int f(int n) { return n == 0 ? 0 : 1 + f(n - 1); }
    public static void main(String[] args) {
        int n = Integer.parseInt(args[0]);
        try { System.out.println("ok at depth " + f(n)); }
        catch (StackOverflowError e) { System.out.println("StackOverflowError"); }
    }
}
"""

if javac is None:
    print("skipped: no JDK (see Part 0)")
else:
    src = os.path.join(workdir, "Depth.java")
    with open(src, "w", encoding="utf-8") as f:
        f.write(JAVA_DEPTH)
    subprocess.run([javac, "-Xlint:all", "-Werror", "-d", workdir, src], check=True,
                   capture_output=True, text=True)
    print("Java, default thread stack:\n")
    for n in [900, 5000, 100_000]:
        got = subprocess.run([_java_exe, "-cp", workdir, "Depth", str(n)],
                             capture_output=True, text=True, check=True).stdout.strip()
        print("  depth %-9s %s" % ("{:,}".format(n), got))

    print()
    print("Java recurses an order of magnitude deeper before failing, and fails with")
    print("StackOverflowError rather than at a configured counter. Neither language")
    print("eliminates tail calls, so neither rescues you: depth is depth.")
    print()
    print("The difference is what you can do about it. Java's limit is a real stack")
    print("size, raisable with -Xss. Python's is an arbitrary counter guarding a")
    print("stack you cannot safely grow.")

Java, default thread stack:

  depth 900       ok at depth 900
  depth 5,000     ok at depth 5000
  depth 100,000   StackOverflowError

Java recurses an order of magnitude deeper before failing, and fails with
StackOverflowError rather than at a configured counter. Neither language
eliminates tail calls, so neither rescues you: depth is depth.

The difference is what you can do about it. Java's limit is a real stack
size, raisable with -Xss. Python's is an arbitrary counter guarding a
stack you cannot safely grow.


***
# Part 2 - The harness every other notebook imports

Part 1 did four things by hand that the remaining 22 notebooks will each need to do repeatedly:
measure growth, decide which class the measurement supports, compare an implementation against a
reference, and run Java from Python.

Doing them by hand every time is how quality bars get quietly dropped. So they live in one
module, **`dsa_toolkit.py`**, which sits beside these notebooks and which every later notebook
imports. This part is a tour of it — what each piece does, and why it is shaped that way.

| Function | What it is for |
|---|---|
| `run_java(source, stdin=...)` | Compile and run a Java program, with `-Xlint:all -Werror` |
| `stress(impl, reference, gen)` | Differential testing, with the failing input minimised |
| `cross_check(py_fn, java_src, gen, to_stdin)` | The same across the two languages |
| `measure_growth(fn, sizes)` | The doubling experiment from §1.4 |
| `fit_complexity(sizes, times)` | Which class do these timings actually support? |
| `growth_table(rows, claim=...)` | Print the table and say whether it matches the claim |
| `check_invariant(structure, predicate)` | Assert the property that defines a structure |
| `edge_cases(kind)` | The adversarial inputs random generation never produces |

In [16]:
# ---------------------------------------------------------------------------
# The module lives beside this notebook. Import it the way every later
# notebook in the series will.
# ---------------------------------------------------------------------------
from dsa_toolkit import (JavaError, StressFailure, InvariantError,
                         check_invariant, cross_check, edge_cases, fit_complexity,
                         growth_table, java_available, measure_growth as tk_measure,
                         run_java, stress)

ok, detail = java_available()
print("JDK available:", ok, "|", detail)

JDK available: True | javac 25.0.4.1


## 2.1 Running Java from Python

The series is bilingual, and the practical question is how to put both languages in one notebook
without splitting every topic into two files.

The answer used here: keep the notebook Python, and hand Java **source** to `run_java`, which
compiles it to a temp directory and runs it. Both implementations then sit in the same cell, a
few lines apart, and the cross-language check is one more line below them.

Two deliberate choices inside it:

- **`-Xlint:all -Werror`.** A Java warning fails the build, exactly as a Python warning fails
  under `warnings.simplefilter("error")`. The series' quality bar applies to both languages.
- **Compilation is cached** by a hash of the source, so re-running a notebook does not recompile.
  Running is *not* cached, and cannot be: each call starts a JVM, which is the floor on cost.

In [17]:
# ---------------------------------------------------------------------------
# A first Java program, and what it costs to run one.
# ---------------------------------------------------------------------------
HELLO = """
public class Adder {
    public static void main(String[] args) {
        java.util.Scanner sc = new java.util.Scanner(System.in);
        System.out.println(sc.nextInt() + sc.nextInt());
    }
}
"""

t0 = time.perf_counter()
first = run_java(HELLO, stdin="2 40\n")
cold = time.perf_counter() - t0

t0 = time.perf_counter()
second = run_java(HELLO, stdin="1 1\n")
warm = time.perf_counter() - t0

print("2 + 40 =", first.strip(), " (%.3fs, compiled)" % cold)
print("1 +  1 =", second.strip(), " (%.3fs, compile cached)" % warm)
print()
print("The cached call still costs %.0f ms, and essentially all of that is JVM" % (warm * 1000))
print("start-up. That sets the budget for cross-language testing: a check over")
print("200 random inputs would cost about %.0f seconds." % (200 * warm))
print("So cross_check defaults to a smaller n than stress does, and Part 3 of")
print("later notebooks batches inputs into a single JVM run when it needs more.")

2 + 40 = 42  (0.115s, compiled)
1 +  1 = 2  (0.113s, compile cached)

The cached call still costs 113 ms, and essentially all of that is JVM
start-up. That sets the budget for cross-language testing: a check over
200 random inputs would cost about 23 seconds.
So cross_check defaults to a smaller n than stress does, and Part 3 of
later notebooks batches inputs into a single JVM run when it needs more.


In [18]:
# ---------------------------------------------------------------------------
# -Werror is not decorative: a lint warning stops the build.
# ---------------------------------------------------------------------------
RAW_TYPE = """
public class Raw {
    public static void main(String[] args) {
        java.util.List items = new java.util.ArrayList();   // raw type: lint warning
        items.add("x");
        System.out.println(items.size());
    }
}
"""

try:
    run_java(RAW_TYPE)
    print("PROBLEM: the warning did not stop the build")
except JavaError as exc:
    lines = [l.strip() for l in str(exc).splitlines() if "warning" in l or "error" in l]
    print("rejected, as intended:")
    for line in lines[:3]:
        print("   ", line[:96])

rejected, as intended:
    C:\Users\abhil\AppData\Local\Temp\dsa_toolkit_java\10f88febc9ee6058\Raw.java:4: warning: [rawtyp
    java.util.List items = new java.util.ArrayList();   // raw type: lint warning
    C:\Users\abhil\AppData\Local\Temp\dsa_toolkit_java\10f88febc9ee6058\Raw.java:4: warning: [rawtyp


## 2.2 Differential testing

The quality bar's first rule: **no implementation ships without being tested against a
reference over randomised inputs**. Not one hand-written example — a thousand generated ones,
plus a fixed set of adversarial cases that random generation essentially never produces.

`stress(impl, reference, gen)` does that, and when it finds a disagreement it **minimises the
input** before reporting it: it repeatedly drops elements while the failure persists. A
counterexample of `[7]` is worth a great deal more than one of 40 random integers.

`edge_cases(kind)` supplies the fixed set: empty, one element, all-equal, already sorted,
reverse sorted, many duplicates, and the 32-bit extremes that make Java overflow.

In [19]:
# ---------------------------------------------------------------------------
# A function with a bug that random testing finds and minimisation explains.
# ---------------------------------------------------------------------------
def second_largest(xs):
    """Return the second largest DISTINCT value, or None. Contains one bug."""
    uniq = sorted(set(xs), reverse=True)
    return uniq[1] if len(uniq) > 1 else uniq[0]      # bug: uniq[0] when len == 0


def second_largest_reference(xs):
    uniq = sorted(set(xs), reverse=True)
    return uniq[1] if len(uniq) > 1 else None


print("adversarial inputs that edge_cases() supplies:")
for case in edge_cases("ints", n=8)[:6]:
    print("   ", case)

print()
try:
    stress(second_largest, second_largest_reference,
           lambda r: [r.randrange(-20, 20) for _ in range(r.randrange(1, 10))],
           n=2000, extra=edge_cases("ints", n=16), label="second_largest")
    print("PROBLEM: the bug was not found")
except StressFailure as exc:
    print("caught:")
    for line in str(exc).splitlines():
        print("   ", line)

print()
print("Note which input it reported. Two thousand random lists all had at least")
print("one element; the empty list came from edge_cases(), and the crash it caused")
print("is the one a real caller would hit first.")

adversarial inputs that edge_cases() supplies:
    []
    [0]
    [1]
    [-1]
    [0, 0]
    [1, 1, 1, 1]

caught:
    second_largest: implementation disagrees with reference
      failing input : []
      implementation: 'impl raised IndexError: list index out of range'
      reference     : None

Note which input it reported. Two thousand random lists all had at least
one element; the empty list came from edge_cases(), and the crash it caused
is the one a real caller would hit first.


In [20]:
# ---------------------------------------------------------------------------
# And the fixed version passes.
# ---------------------------------------------------------------------------
def second_largest_fixed(xs):
    uniq = sorted(set(xs), reverse=True)
    return uniq[1] if len(uniq) > 1 else None


checked = stress(second_largest_fixed, second_largest_reference,
                 lambda r: [r.randrange(-20, 20) for _ in range(r.randrange(0, 10))],
                 n=2000, extra=edge_cases("ints", n=16), label="second_largest_fixed")
print("%s cases, implementation and reference agree on every one." % "{:,}".format(checked))

2,014 cases, implementation and reference agree on every one.


## 2.3 Cross-language checking

The rule that makes this series bilingual rather than merely bi-lingual-looking: **the Python and
Java implementations are run on identical inputs and must produce identical output.**

This catches a whole class of bug that neither implementation reveals on its own — integer
overflow, integer division and modulo on negatives, sort stability, iteration order. The cell
below demonstrates the first one, which is the single most common Java bug in this material and
is invisible from Python because Python integers do not overflow.

In [21]:
# ---------------------------------------------------------------------------
# Sum a list. Python and Java, same inputs.
# ---------------------------------------------------------------------------
JAVA_SUM_LONG = """
public class SumLong {
    public static void main(String[] args) {
        java.util.Scanner sc = new java.util.Scanner(System.in);
        int n = sc.nextInt();
        long total = 0;                       // 64-bit accumulator
        for (int i = 0; i < n; i++) total += sc.nextInt();
        System.out.println(total);
    }
}
"""


def to_stdin(xs):
    return "%d %s\n" % (len(xs), " ".join(map(str, xs)))


n = cross_check(sum, JAVA_SUM_LONG,
                lambda r: [r.randrange(-10 ** 6, 10 ** 6) for _ in range(r.randrange(1, 12))],
                to_stdin, n=12, label="sum with long")
print("%d random inputs: Python and Java agree every time.\n" % n)

# The same program with a 32-bit accumulator.
JAVA_SUM_INT = JAVA_SUM_LONG.replace("long total", "int total").replace("SumLong", "SumInt")

try:
    cross_check(sum, JAVA_SUM_INT, lambda r: [2_000_000_000, 2_000_000_000],
                to_stdin, n=1, label="sum with int")
    print("PROBLEM: overflow not detected")
except StressFailure as exc:
    print("one line changed -- long became int:")
    for line in str(exc).splitlines():
        print("   ", line)

print()
print("Python's integers grow without bound, so it reports 4,000,000,000. Java's")
print("int is 32 bits and wraps to a NEGATIVE number. Neither implementation looks")
print("wrong on its own; only running both on the same input shows it.")
print()
print("NB-15 meets this again as a genuine bug in production code: the standard")
print("binary search midpoint (lo + hi) / 2 overflows in Java, and the JDK shipped")
print("that bug for nine years.")

12 random inputs: Python and Java agree every time.

one line changed -- long became int:
    sum with int: Python and Java disagree
      input : [2000000000, 2000000000]
      python: 4000000000
      java  : -294967296

Python's integers grow without bound, so it reports 4,000,000,000. Java's
int is 32 bits and wraps to a NEGATIVE number. Neither implementation looks
wrong on its own; only running both on the same input shows it.

NB-15 meets this again as a genuine bug in production code: the standard
binary search midpoint (lo + hi) / 2 overflows in Java, and the JDK shipped
that bug for nine years.


## 2.4 Invariants, asserted rather than described

Every data structure in this series is introduced by the **invariant** it maintains — the property
that is true before and after every operation, and that makes the structure what it is.

Saying "a heap maintains the heap property" is prose. Calling `check_invariant` after every push
and pop in a randomised sequence is a verified claim, and it is the difference between a notebook
that looks right and one that is.

In [22]:
# ---------------------------------------------------------------------------
# A tiny sorted-list structure, with its invariant checked after every write.
# ---------------------------------------------------------------------------
import bisect


def is_sorted(xs):
    """The invariant. Returns True, or a string explaining the violation."""
    for i in range(1, len(xs)):
        if xs[i - 1] > xs[i]:
            return "xs[%d]=%r > xs[%d]=%r" % (i - 1, xs[i - 1], i, xs[i])
    return True


rng2 = random.Random(RANDOM_SEED)
values = []
for step in range(2000):
    v = rng2.randrange(-50, 50)
    bisect.insort(values, v)                                    # the operation
    check_invariant(values, is_sorted, "sorted order", "insort(%d)" % v)
print("2,000 insertions, invariant held after every one. len = %d" % len(values))

# Now break it deliberately, to show the failure is informative.
values[len(values) // 2] = 999
try:
    check_invariant(values, is_sorted, "sorted order", "a corrupting write")
    print("PROBLEM: violation not detected")
except InvariantError as exc:
    print()
    print("deliberately corrupted:")
    for line in str(exc).splitlines()[:2]:
        print("   ", line[:100])

2,000 insertions, invariant held after every one. len = 2000

deliberately corrupted:
    sorted order violated after a corrupting write: xs[1000]=999 > xs[1001]=-2
      state: [-50, -50, -50, -50, -50, -50, -50, -50, -50, -50, -50, -50, -50, -50, -50, -50, -50, -50, 


## 2.5 Deciding what the timings actually support

§1.4 read complexity classes off the ratio column by eye. `fit_complexity` does it numerically:
for each candidate $f$, it fits the single scale factor $c$ minimising the **relative** error of
$c\,f(n)$ against the measurements, and ranks the candidates.

Relative, not absolute — otherwise the largest size dominates the fit entirely and the answer is
whatever explains the last row.

`growth_table(rows, claim=...)` prints the table, reports the best fit **and the runner-up**, and
says whether the measurement supports the claim. The runner-up matters: when the top two are
close, the honest conclusion is that this experiment cannot separate them. §3.2 is that case.

In [23]:
# ---------------------------------------------------------------------------
# The whole loop: measure, then let the fit decide.
# ---------------------------------------------------------------------------
print("A binary search, claimed O(log n):\n")


def bsearch(payload):
    arr, target = payload
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        if arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1


def many_searches(payload):
    arr, targets = payload
    for t in targets:
        bsearch((arr, t))


growth_table(tk_measure(
    many_searches, [200_000, 400_000, 800_000, 1_600_000], repeats=3,
    setup=lambda n: (list(range(n)),
                     [random.Random(1).randrange(n) for _ in range(2000)])),
    claim="O(log n)")

print()
print("2,000 searches at each size, so the per-search cost is what grows. Doubling")
print("the array adds one comparison per search, which is why the ratio hovers")
print("near 1 rather than 2.")

A binary search, claimed O(log n):

         n        seconds      ratio
------------------------------------
   200,000       0.007824          -
   400,000       0.009792       1.25
   800,000       0.007012       0.72
 1,600,000       0.010526       1.50

best fit: O(log n) (relative error 0.165); next: O(1) (0.171)
claimed O(log n) -> measurement MATCHES the claim

2,000 searches at each size, so the per-search cost is what grows. Doubling
the array adds one comparison per search, which is why the ratio hovers
near 1 rather than 2.


***
# Part 3 - When measurement and theory disagree

Everything above argued for measuring. This part is about the three ways measurement misleads,
because a habit you cannot criticise is not a discipline.

## 3.1 The benchmark that measures the wrong distribution

§1.6 showed insertion sort measuring as $O(n^2)$ on random input and $O(n)$ on sorted input. Both
measurements are correct. Either one, reported alone as "insertion sort is …", is a lie.

The failure mode is easy to fall into by accident, and the cell below shows it happening from a
one-character mistake in the *generator* rather than in the algorithm.

In [24]:
# ---------------------------------------------------------------------------
# Two generators that look equivalent. One of them destroys the experiment.
# ---------------------------------------------------------------------------
def per_element_seed(n):
    """Looks random. Re-seeds for every element, so every value is IDENTICAL."""
    return [random.Random(n).randrange(10 ** 6) for _ in range(n)]


def seeded_once(n):
    """Actually random, and reproducible."""
    r = random.Random(n)
    return [r.randrange(10 ** 6) for _ in range(n)]


bad, good = per_element_seed(8), seeded_once(8)
print("per-element seeding ->", bad)
print("  distinct values:", len(set(bad)))
print("seeded once         ->", good)
print("  distinct values:", len(set(good)))
print()

show(measure_growth(insertion_sort, [400, 800, 1600, 3200], repeats=3,
                    setup=per_element_seed),
     "insertion sort on the BROKEN generator -> looks like O(n)")

show(measure_growth(insertion_sort, [400, 800, 1600, 3200], repeats=3,
                    setup=seeded_once),
     "insertion sort on real random data -> O(n^2), as it should be")

print("An all-equal list is insertion sort's best case, so the broken generator")
print("measured the best case and called it the average. The algorithm was never")
print("wrong; the experiment was.")
print()
print("This exact bug produced a first draft of section 1.6 in which insertion sort")
print("beat merge sort at every size up to 6,400 -- a result that should have been")
print("unbelievable, and was.")
print()
print("The defence is a sanity check you can state in advance: an O(n^2) algorithm")
print("MUST show a ratio near 4. When it does not, suspect the harness before you")
print("believe the headline.")

per-element seeding -> [237718, 237718, 237718, 237718, 237718, 237718, 237718, 237718]
  distinct values: 1
seeded once         -> [237718, 388404, 393603, 132467, 202497, 739054, 45905, 89323]
  distinct values: 8

  insertion sort on the BROKEN generator -> looks like O(n)
           n       seconds     ratio
  -----------------------------------
         400      0.000064         -
         800      0.000136      2.12
       1,600      0.000280      2.06
       3,200      0.000568      2.03

  insertion sort on real random data -> O(n^2), as it should be
           n       seconds     ratio
  -----------------------------------
         400      0.006311         -
         800      0.027048      4.29
       1,600      0.109258      4.04
       3,200      0.444814      4.07

An all-equal list is insertion sort's best case, so the broken generator
measured the best case and called it the average. The algorithm was never
wrong; the experiment was.

This exact bug produced a first draf

## 3.2 A verdict that changes when you run it again

Merge sort is $\Theta(n \log n)$. Measure it over a modest range of sizes and the fit will tell
you so — sometimes.

The problem is not that the answer is wrong. It is that **the answer is not reproducible**: run
the identical experiment eight times and the winner changes. That is worse than a wrong answer,
because a single run looks perfectly convincing either way.

The cause is that between $n$ and $2n$ the $\log n$ factor moves the ratio from 2.00 to about
2.1, and at these sizes each run takes only milliseconds, so timing noise is larger than the
effect being measured.

In [25]:
# ---------------------------------------------------------------------------
# Run the SAME narrow-range experiment eight times and collect the verdicts.
# ---------------------------------------------------------------------------
NARROW = [250, 500, 1000, 2000, 4000]

print("Merge sort over n = 250 .. 4,000, eight independent repetitions:\n")
print("  %-4s %-22s %-22s" % ("run", "best fit", "runner-up"))
print("  " + "-" * 50)
verdicts = []
for run in range(8):
    rows = tk_measure(merge_sort, NARROW, repeats=3,
                      setup=lambda n: random.Random(run * 97 + n).sample(range(10 ** 7), n))
    ranked = fit_complexity(NARROW, [r["seconds"] for r in rows])
    verdicts.append(ranked[0][0])
    print("  %-4d %-22s %-22s" % (run, "%s %.3f" % ranked[0], "%s %.3f" % ranked[1]))

tally = Counter(verdicts)
print()
print("  verdicts: %s" % dict(tally))
print()
if len(tally) > 1:
    print("  The same experiment, run eight times, does not agree with itself.")
    print("  A single run of this would have looked entirely convincing.")
else:
    print("  Stable on this machine today -- but see the error columns: the two")
    print("  candidates are close enough that a slower machine would flip it.")

Merge sort over n = 250 .. 4,000, eight independent repetitions:

  run  best fit               runner-up             
  --------------------------------------------------
  0    O(n log n) 0.057       O(n) 0.173            
  1    O(n log n) 0.073       O(n) 0.143            
  2    O(n log n) 0.064       O(n) 0.182            
  3    O(n log n) 0.020       O(n) 0.137            
  4    O(n log n) 0.050       O(n) 0.168            
  5    O(n log n) 0.058       O(n) 0.188            
  6    O(n log n) 0.051       O(n) 0.168            
  7    O(n log n) 0.062       O(n) 0.167            

  verdicts: {'O(n log n)': 8}

  Stable on this machine today -- but see the error columns: the two
  candidates are close enough that a slower machine would flip it.


In [26]:
# ---------------------------------------------------------------------------
# Is the fitter to blame? Give it EXACT data with no timing noise at all.
# ---------------------------------------------------------------------------
sizes = [1000, 4000, 16000, 64000, 256000]
exact_linear = [float(n) for n in sizes]
exact_nlogn = [n * math.log(n, 2) for n in sizes]

for label, data in [("exact O(n) data     ", exact_linear),
                    ("exact O(n log n) data", exact_nlogn)]:
    top, second = fit_complexity(sizes, data)[:2]
    print("  %s -> %-12s %.4f   (next: %-12s %.4f)"
          % (label, top[0], top[1], second[0], second[1]))
print()
print("Exact separation: zero error for the right class, and a clear gap to the")
print("wrong one. The fitter can tell them apart perfectly well.")
print()
print("So the instability above was TIMING NOISE, at sizes where each run takes")
print("a few milliseconds. Not a statistics problem -- a measurement problem.")

  exact O(n) data      -> O(n)         0.0000   (next: O(n log n)   0.2160)
  exact O(n log n) data -> O(n log n)   0.0000   (next: O(n)         0.2255)

Exact separation: zero error for the right class, and a clear gap to the
wrong one. The fitter can tell them apart perfectly well.

So the instability above was TIMING NOISE, at sizes where each run takes
a few milliseconds. Not a statistics problem -- a measurement problem.


In [27]:
# ---------------------------------------------------------------------------
# The fix is not a better statistic. It is bigger inputs.
# ---------------------------------------------------------------------------
print("The same merge sort, n = 2,000 .. 512,000:\n")
growth_table(tk_measure(merge_sort, [2000, 8000, 32000, 128000, 512000], repeats=3,
                        setup=lambda n: random.Random(7).sample(range(10 ** 8), n)),
             claim="O(n log n)")

print()
print("Now it resolves, and the runner-up is clearly behind. Repeat this one and")
print("it keeps saying the same thing -- which is the property the narrow-range")
print("experiment lacked.")
print()
print("The rule this gives you: a doubling experiment needs runs long enough that")
print("noise is a small fraction of the measurement -- tens of milliseconds at")
print("least -- and a range wide enough for the terms to separate. Two adjacent")
print("sizes and a stopwatch prove nothing.")
print()
print("And the cheap defence, whatever the range: RUN IT TWICE. A verdict that")
print("moves between runs is not a verdict.")
print()
print("And when you cannot get there, COUNT instead of timing (1.7). The comparison")
print("count matched n*log2(n)-n+1 to three decimal places at n=16, where timing")
print("could not have told you anything at all.")

The same merge sort, n = 2,000 .. 512,000:

         n        seconds      ratio
------------------------------------
     2,000       0.006466          -
     8,000       0.034156       5.28
    32,000       0.164437       4.81
   128,000       0.757265       4.61
   512,000       3.378345       4.46

best fit: O(n log n) (relative error 0.066); next: O(n) (0.290)
claimed O(n log n) -> measurement MATCHES the claim

Now it resolves, and the runner-up is clearly behind. Repeat this one and
it keeps saying the same thing -- which is the property the narrow-range
experiment lacked.

The rule this gives you: a doubling experiment needs runs long enough that
noise is a small fraction of the measurement -- tens of milliseconds at
least -- and a range wide enough for the terms to separate. Two adjacent
sizes and a stopwatch prove nothing.

And the cheap defence, whatever the range: RUN IT TWICE. A verdict that
moves between runs is not a verdict.

And when you cannot get there, COUNT instead

## 3.3 When the bound is right and irrelevant

The third failure is the one that sends teams optimising the wrong thing: the bound is correct,
and it describes a case your input never reaches.

- **Hash table lookup is $O(n)$ worst case.** It is $O(1)$ in practice, always, until someone
  feeds you deliberately colliding keys — NB-03 constructs exactly that input and measures the
  collapse.
- **Quicksort is $\Theta(n^2)$ worst case.** Randomising the pivot makes that case
  astronomically unlikely rather than impossible, which is why every library quicksort does it.
- **Union-Find is $O(\alpha(n))$ amortised**, where $\alpha$ is the inverse Ackermann function.
  It is below 5 for any input that fits in the observable universe, so the honest summary is
  "constant", and NB-11 measures it being constant.

The discipline that follows:

1. **Know the worst case exists** and what triggers it. That is a design question, not a
   benchmarking one.
2. **Measure the case you will actually see**, and say which case you measured.
3. **Ask whether an adversary chooses your input.** If yes — user-supplied keys, untrusted data —
   the worst case is not a tail risk, it is a threat model.

In [28]:
# ---------------------------------------------------------------------------
# The gap between a worst case and a realistic one, on dictionary lookup.
# ---------------------------------------------------------------------------
print("Python dict lookup, ordinary integer keys:\n")


def lookups(payload):
    d, keys = payload
    for k in keys:
        _ = d[k]


def setup_normal(n):
    d = {i: i for i in range(n)}
    keys = [random.Random(3).randrange(n) for _ in range(20_000)]
    return d, keys


growth_table(tk_measure(lookups, [50_000, 100_000, 200_000, 400_000], repeats=3,
                        setup=setup_normal), claim="O(1)")

print()
print("20,000 lookups at every size, and the total barely moves as the dictionary")
print("grows 8x. That is the O(1) everyone relies on.")
print()
print("The O(n) worst case is real and requires keys that all hash to one bucket.")
print("NB-03 builds those keys and measures what happens -- it is not subtle, and")
print("it is a genuine denial-of-service vector, which is why Python randomises")
print("string hashing per process by default.")

Python dict lookup, ordinary integer keys:

         n        seconds      ratio
------------------------------------
    50,000       0.001158          -
   100,000       0.001072       0.93
   200,000       0.001094       1.02
   400,000       0.001111       1.02

best fit: O(1) (relative error 0.028); next: O(2^n) (0.028)
claimed O(1) -> measurement MATCHES the claim

20,000 lookups at every size, and the total barely moves as the dictionary
grows 8x. That is the O(1) everyone relies on.

The O(n) worst case is real and requires keys that all hash to one bucket.
NB-03 builds those keys and measures what happens -- it is not subtle, and
it is a genuine denial-of-service vector, which is why Python randomises
string hashing per process by default.


***
# Part 4 - Tough questions

***

### Q1. What does $O(n)$ actually mean, and how is it different from $\Theta(n)$?

<details><summary>Answer</summary>

$T(n) = O(f(n))$ means there are constants $c > 0$ and $n_0$ with $T(n) \le c\,f(n)$ for all
$n \ge n_0$. It is an **upper bound** on the growth rate, valid only in the tail, with the
constant quantified away.

| | Bound | Says |
|---|---|---|
| $O(f)$ | upper | grows no faster than $f$ |
| $\Omega(f)$ | lower | grows no slower than $f$ |
| $\Theta(f)$ | both | grows exactly like $f$ |

Because $O$ is only an upper bound, "merge sort is $O(n^2)$" is a **true** statement — and
useless. When you mean tight, say $\Theta$.

**Two consequences people miss:**

- **The constant is gone by construction.** §1.3 measures the same $O(n)$ loop running **hundreds
  of times** faster in Java than in Python. Both are $O(n)$; the notation cannot see the difference, and
  your latency budget can.
- **It is a claim about the tail.** §1.6 finds an $O(n^2)$ sort beating an $O(n\log n)$ sort for
  every $n$ below **100**. "Eventually" is a real number, and for a function called on ten
  elements it may never arrive.

**Best/average/worst are separate claims**, and $O$/$\Omega$/$\Theta$ can be applied to each.
Saying "quicksort is $O(n \log n)$" conflates the average with the worst case, which is
$\Theta(n^2)$.

</details>

***

### Q2. How do you determine an algorithm's complexity by measurement?

<details><summary>Answer</summary>

**The doubling experiment** (§1.4). Run at sizes that double and look at the ratio of consecutive
times. The constant cancels, so the ratio alone identifies the class:

$$ \frac{T(2n)}{T(n)} \approx \frac{f(2n)}{f(n)} $$

| Ratio | Class |
|---|---|
| ≈ 1 | $O(1)$ |
| slightly above 1, falling | $O(\log n)$ |
| ≈ 2 | $O(n)$ |
| just over 2 | $O(n \log n)$ |
| ≈ 4 | $O(n^2)$ |
| ≈ 8 | $O(n^3)$ |

**Four details that decide whether the experiment is worth anything:**

1. **Take the minimum of several runs, not the mean.** Noise is one-sided — an interrupt or a GC
   pause can only make a run slower — so the minimum is the best estimate of the true cost.
2. **Exclude setup.** Building the input is not the thing being measured.
3. **Use big enough sizes.** §3.2 runs merge sort over $n = 250..4000$ eight times and gets
   two different answers; over $n = 2000..512{,}000$ it gets the same answer every time. The
   first experiment was not wrong statistics, it was too small — each run took milliseconds,
   and noise swamped the $\log n$ term.
4. **Use a wide enough range.** Adjacent sizes cannot separate anything.

**And when you can, count instead of timing.** §1.7 counts merge sort's comparisons and matches
$n\log_2 n - n + 1$ to three decimal places at $n = 16$ — no noise, no constants, identical on
every machine. Timing is the fallback for when counting is not available.

</details>

***

### Q3. What is amortised analysis, and how is it different from average-case?

<details><summary>Answer</summary>

**Amortised analysis is the average cost per operation over a worst-case sequence.** Average-case
analysis is a probabilistic claim about random input. They are different in the way that matters:

| | Average case | Amortised |
|---|---|---|
| Assumes | inputs are drawn from some distribution | **nothing** |
| Guarantee over | typical inputs | **any sequence, chosen adversarially** |
| Fails when | your data is not typical | never |

A hash table lookup is $O(1)$ **average** — an adversary picking colliding keys breaks it. A
dynamic-array append is $O(1)$ **amortised** — no adversary can make $n$ appends cost more than
$O(n)$ total, because the argument does not depend on the values at all.

**Three techniques, same answer:**

- **Aggregate.** $n$ appends with doubling copy $1+2+4+\dots+n < 2n$ elements, so $O(n)$ total.
- **Accounting.** Charge 3 units per append: 1 to store, 2 banked toward the future copy. The
  bank is never negative, so the amortised cost is $O(1)$.
- **Potential.** $\Phi = 2\cdot\text{size} - \text{capacity}$; a cheap append raises it, a resize
  spends it, and actual + $\Delta\Phi$ stays constant.

**The load-bearing requirement is geometric growth.** §1.5 measures CPython's list growing by a
settled factor of about **1.125×** — 66 reallocations across 100,000 appends — and Java's
`ArrayList` uses 1.5×. Replace the factor with "grow by one" and the same interface becomes
$O(n)$ per append and $O(n^2)$ overall; §1.5 measures that too, at a ratio of ~4.

**Where it appears later:** dynamic arrays (NB-01), the two-stack queue (NB-05), and Union-Find's
$\alpha(n)$ bound (NB-11), which is amortised and not worst case.

</details>

***

### Q4. Your algorithm is $O(n \log n)$ and a colleague's is $O(n^2)$. Theirs is faster. Explain.

<details><summary>Answer</summary>

Nothing is wrong; several ordinary things could be true, and §1.6 measures the first two.

1. **You are below the crossover.** Constants differ, and asymptotics only claim a tail. Measured
   here: insertion sort beats merge sort for all $n < 100$, and at $n = 50$ it wins outright.
   Production sorts exploit this — Timsort and introsort both switch to insertion sort on short
   runs.
2. **The input is not the average case.** Insertion sort is $\Theta(n)$ on sorted or nearly-sorted
   data, so on that input it is *asymptotically* faster, not just constant-factor faster.
   §1.6 measures it beating merge sort by several times at $n = 8000$ and staying ahead.
3. **Memory behaviour.** The $O(n^2)$ algorithm may be scanning contiguous memory while yours
   chases pointers. §1.2 measures a several-fold penalty for random over sequential access on
   the same data, in Python, where interpreter overhead should have hidden it.
4. **Your constant is genuinely bad.** Recursion, allocation per level, and copying can cost more
   than the extra factor of $n/\log n$ saves at your sizes.

**What to do:** find the crossover, then use it. Hybrid algorithms exist precisely because the
answer is "both, at different sizes". And confirm which case you are measuring — §3.1 shows a
one-character generator bug turning insertion sort's best case into an apparent average case.

</details>

***

### Q5. Explain the Master theorem and use it on merge sort.

<details><summary>Answer</summary>

For $T(n) = a\,T(n/b) + f(n)$ — $a$ subproblems, each of size $n/b$, plus $f(n)$ work to split
and combine — compare $f(n)$ with $n^{\log_b a}$, which counts the leaves.

| Case | Condition | Result |
|---|---|---|
| 1 | $f(n) = O(n^{\log_b a - \epsilon})$ | $\Theta(n^{\log_b a})$ — leaves dominate |
| 2 | $f(n) = \Theta(n^{\log_b a})$ | $\Theta(n^{\log_b a}\log n)$ — balanced |
| 3 | $f(n) = \Omega(n^{\log_b a + \epsilon})$ + regularity | $\Theta(f(n))$ — root dominates |

**Merge sort:** $a = 2$, $b = 2$, $f(n) = \Theta(n)$. So $n^{\log_2 2} = n^1 = n$, which equals
$f(n)$ — **case 2** — giving $\Theta(n \log n)$.

**The intuition, and it is the part worth keeping:** the recursion tree has $\log_b n$ levels;
the case is decided by whether per-level work grows, shrinks, or stays flat going down. §1.7
counts the work at every level for all three cases at $n = 1024$, and exactly one of
$\text{total}/n$, $\text{total}/(n\log n)$, $\text{total}/n^2$ comes out a small constant in each
case — 2.00, 1.10 and 2.00 respectively.

**Others worth recognising:** binary search $T(n) = T(n/2) + O(1) \Rightarrow \Theta(\log n)$;
naive matrix multiply $T(n) = 8T(n/2) + O(n^2) \Rightarrow \Theta(n^3)$; Strassen
$T(n) = 7T(n/2) + O(n^2) \Rightarrow \Theta(n^{\log_2 7}) \approx \Theta(n^{2.81})$ — the same
recurrence shape, one fewer multiplication, a different exponent.

**When it does not apply:** unequal subproblem sizes ($T(n) = T(n/3) + T(2n/3) + n$), non-constant
$a$ or $b$, or an $f$ that fits between the cases. Then use the recursion-tree method directly,
or substitution.

</details>

***

### Q6. What is space complexity, and what do people forget to count?

<details><summary>Answer</summary>

The same analysis applied to memory. The two things routinely missed:

**1. The recursion stack.** Every pending call holds a frame. A recursion $n$ deep costs $O(n)$
space even if it allocates nothing. §1.8 measures the ceiling: Python raises `RecursionError`
past its default limit of **1000**, so a recursive traversal of a 5,000-node degenerate tree
crashes; Java survives 5,000 and fails with `StackOverflowError` around 100,000.

Neither language eliminates tail calls, so restructuring to a loop or an explicit stack is the
only fix — NB-06 does exactly that for tree traversal.

**2. Auxiliary vs total.** "In place" is a claim about *auxiliary* space, the working memory
beyond the input. Merge sort needs $O(n)$ auxiliary; quicksort needs $O(\log n)$ for its stack
and is called in-place; heapsort needs $O(1)$.

**Where the trade-off is the point:**

- **Memoisation** (NB-19) buys time with space — that is the entire technique.
- **Tries** (NB-10) buy $O(\text{key length})$ lookup with substantial memory overhead, measured
  there against a plain `set`.
- **Adjacency matrix vs list** (NB-20): $O(V^2)$ against $O(V+E)$, with the crossover measured.

**A caution on `sys.getsizeof`:** it reports the object's own size, not what it references. A
list of a million integers reports about 8 MB — the pointer array — while the integers themselves
are separate objects. §1.5 uses it correctly, to observe the *buffer* growing, not the total.

</details>

***

### Q7. Why can two $O(n)$ implementations differ by 100× in practice?

<details><summary>Answer</summary>

Because $O(n)$ discards exactly the information that decides it. Four sources, in rough order of
how often they bite:

**1. Language and runtime.** §1.3 measures the identical summing loop **hundreds of times** apart
between Python and Java. Same algorithm, same answer, same complexity class.

**2. Memory access pattern.** The RAM cost model assumes uniform memory; real machines have a
cache hierarchy spanning about two orders of magnitude. §1.2 measures sequential, strided and
random traversal of one list, and the random order costs several times more — through Python's
interpreter overhead, which should have masked it. In compiled code the gap is larger. This is
most of why arrays beat linked lists (NB-04) despite matching complexities.

**3. Constant work per element.** Allocating an object per iteration, boxing an `int` into an
`Integer`, or copying a slice all multiply the constant without touching the class.

**4. What the compiler does.** JIT compilation, bounds-check elimination and vectorisation are
worth large factors and are not visible in the source. §1.3's Java loop is JIT-compiled after a
few thousand iterations; a loop that ran 100 times would not be.

**The practical upshot:** use Big-O to choose the algorithm — you cannot benchmark your way to
knowing quicksort beats bubble sort at scale — then profile to find the constant. In that order.
Choosing the wrong algorithm cannot be optimised away; a bad constant usually can.

</details>

***

### Q8. What is the difference between $O(1)$ and $O(\log n)$ in practice?

<details><summary>Answer</summary>

Much less than the notation suggests, and this is one of the few places where the theory
overstates a difference.

$\log_2 n$ for realistic $n$:

| $n$ | $\log_2 n$ |
|---|---|
| 1,000 | 10 |
| 1,000,000 | 20 |
| 1,000,000,000 | 30 |
| $10^{12}$ | 40 |

**A logarithm is bounded by a small constant across every input size that exists.** Going from a
thousand to a trillion elements multiplies the work by four. So an $O(\log n)$ structure with a
small constant routinely beats an $O(1)$ structure with a large one.

**Concretely, and this recurs through the series:** a hash table is $O(1)$ and a balanced BST is
$O(\log n)$, yet `TreeMap` is competitive with `HashMap` at small sizes and wins outright when you
need ordering, because getting a range or the next-largest key from a hash table means scanning
all of it. NB-08 measures that comparison.

**Where the difference does bite:** in the innermost loop of something already fast, run
billions of times — and when the $O(\log n)$ operation is a **cache miss per level**, as pointer
chasing through a tree is. Then the 20 levels are 20 trips to main memory, and that is a real
cost the notation does not show.

**The honest summary:** treat $O(\log n)$ as "effectively constant, with a worse constant", and
choose on what the structure can *do* — ordering, ranges, predecessors — rather than on the
logarithm.

</details>

***

### Q9. How would you empirically verify a complexity claim?

<details><summary>Answer</summary>

In order of how convincing the evidence is:

**1. Count operations.** The strongest, when the thing you care about is countable. §1.7
instruments merge sort's comparison counter and compares against $n\log_2 n - n + 1$: the ratio
sits at 0.96–0.98 and rises toward 1. No noise, no constants, no machine dependence.

**2. Run the doubling experiment.** Sizes that double, minimum of several runs, setup excluded,
read the ratio (Q2). Large enough sizes that each run takes tens of milliseconds — §3.2 shows
what happens otherwise.

**3. Fit, report the runner-up, and run it twice.** `fit_complexity` ranks candidates by
relative error; when the top two are close the experiment cannot separate them. The cheapest
check of all is repetition — §3.2 runs one experiment eight times and gets two different
verdicts, which no single run would have revealed.

**Four failure modes to check for before believing your own result:**

- **The generator is broken.** §3.1: re-seeding an RNG per element produced an all-equal list,
  which is insertion sort's best case, which made an $O(n^2)$ algorithm measure as $O(n)$.
- **You measured the wrong case.** State which distribution you used.
- **Sizes too small.** Noise dominates, and the verdict changes between runs (§3.2).
- **Something else dominates.** Setup, I/O, or the first JIT-interpreted iterations.

**The sanity check worth doing every time:** you know roughly what ratio the claim predicts —
2 for linear, 4 for quadratic. If the measurement is far off, suspect the harness before you
believe the headline.

</details>

***

### Q10. When is an exponential algorithm acceptable?

<details><summary>Answer</summary>

More often than the horror stories suggest, and always for one of these reasons:

**1. $n$ is small and bounded.** $2^{20}$ is a million — instant. Bitmask DP over subsets
(NB-19) is $O(2^n \cdot n)$ and entirely practical to about $n = 20$–25. If your input is a
chessboard or a 15-city tour, exponential is fine and the code is simple.

**2. Pruning changes the effective base.** Backtracking (NB-17) is exponential in the worst case
and routinely fast in practice because most branches die early. N-Queens with constraint checks
explores a small fraction of the nominal tree — NB-17 measures the node counts with and without
pruning rather than asserting the improvement.

**3. The problem is NP-hard and you need the exact answer.** Then exponential is the price, and
the alternatives are approximation with a proven ratio, a heuristic with no guarantee, or an
ILP/SAT solver — which is still exponential in the worst case but very good at avoiding it.

**4. It is a fallback.** Fast path for the common case, exact exponential for rare hard
instances.

**When it is not acceptable:** unbounded or user-controlled $n$. That is a denial-of-service
vector, not a performance problem. Catastrophic regex backtracking is the classic: an innocent
pattern going exponential on a crafted input, and it has taken down major sites.

**The number that settles it:** §1.1's table shows $2^n$ at $n = 100$ exceeding anything
physically computable. Below ~25 it is nothing. There is almost no middle ground, which is why
the decision is usually easy once you know your bound on $n$.

</details>

***

### Q11. You are told a service "got slower after we doubled the data". How do you find out why?

<details><summary>Answer</summary>

The doubling is the gift here: it turns a vague complaint into a measurement.

**1. Get the ratio.** How much slower for 2× the data? That number names the suspect directly
(Q2): ~2 is linear and probably just growth; ~4 says something is quadratic; a jump far worse
than 4 suggests thrashing, swapping, or falling out of cache rather than an algorithm at all.

**2. A ratio near 1 is informative too** — it means the slow part does not scale with the data,
so look at fixed costs: a cold cache, a connection pool, an $N{+}1$ query pattern that got slower
for an unrelated reason.

**3. Suspect the quadratic accident.** In real code $O(n^2)$ is rarely a nested loop someone
wrote deliberately. It is usually:
   - a linear scan inside a loop (`if x in list` where `list` is long — use a set),
   - string concatenation in a loop (NB-02 measures this),
   - `list.pop(0)` or `insert(0, …)` in a loop, each $O(n)$ (NB-05: use a deque),
   - a "grow by one" array policy (§1.5, ratio ~4).

**4. Check whether the *distribution* changed, not just the size.** §1.6 and §3.1: the same
algorithm can change complexity class when its input becomes sorted, or clustered, or full of
duplicates. More data often means differently-shaped data.

**5. Then profile**, to confirm rather than to explore. Profiling first tells you where the time
goes without telling you why it grew.

**What to build so the next one is easier:** record timing against input size in production. With
that series, the ratio is a query rather than an investigation.

</details>

***

### Q12. Is optimising for Big-O always the right priority?

<details><summary>Answer</summary>

No, and treating it as the priority is its own failure mode.

**When the complexity class is the whole game:** unbounded input, an inner loop, or a difference
of a factor of $n$. No constant-factor work rescues a quadratic algorithm on a million elements —
§1.1 puts it at about 17 minutes against 0.02 seconds. Choose the algorithm first, because that
choice cannot be optimised away later.

**When it is the wrong priority:**

- **$n$ is small and stays small.** §1.6: below 100 elements the $O(n^2)$ sort is the faster one.
  Optimising a function that sorts ten items is not engineering.
- **The constant is the problem.** §1.3's language gap dwarfs most algorithmic wins at
  moderate $n$.
- **It is not the bottleneck.** A function taking 2% of your runtime cannot give you more than
  2%, however good it becomes.
- **The cost is clarity.** A clever $O(n)$ solution that the team cannot modify safely is a
  liability. `sorted(xs)[k]` is $O(n \log n)$ where quickselect is $O(n)$, and for almost every
  real call site it is the better line of code.
- **The bottleneck is elsewhere entirely.** A network round trip is ~100,000× a memory access.
  Removing one query beats any amount of in-memory tuning.

**The order that works:** get it correct, measure to find where the time is, fix the algorithm if
the algorithm is the problem, then the constant, then stop. §3.3's point applies throughout —
know that the worst case exists, know what triggers it, and only optimise for it if an adversary
picks your input.

</details>

***

## Coding challenges

### Challenge 1 — a complexity detector

`fit_complexity` ranks candidate classes by relative error. Make it say how confident it is.

1. Add a bootstrap: resample the (size, time) pairs with replacement, refit, and record the
   winner each time. Report the fraction of resamples each class won.
2. Use it on §3.2's narrow-range merge-sort data. It should report genuine uncertainty between
   $O(n)$ and $O(n\log n)$ rather than a single answer.
3. Add a check that refuses to answer at all when the range of sizes is under 8×, or when the
   fastest run is under a millisecond — the two conditions §3.2 identified.
4. Test it against functions of known complexity, including one you construct to be $O(n^{1.5})$.
   Does the fitter cope with a class that is not in its list? Should it say so?

***

### Challenge 2 — the accounting method, in code

§1.5 argued amortised $O(1)$ three ways. Verify the accounting argument mechanically.

1. Implement a dynamic array with an explicit `credits` counter. Charge 3 units per append: spend
   1 on the store, bank 2.
2. On a resize, spend banked credits — one per element copied. Assert after every operation that
   the balance is **never negative**. That assertion *is* the proof.
3. Run it for 100,000 appends with a growth factor of 2, then 1.5, then 1.125. What is the
   minimum charge per append that keeps the balance non-negative for each factor? Derive the
   relationship and check it against your measurements.
4. Now set the growth factor to 1.0 (grow by one). Watch the balance go negative, and say
   precisely which step of the argument fails.

***

### Challenge 3 — cross-language cost model

§1.3 measured one loop at a few hundred times. One data point is an anecdote.

1. Build a small benchmark suite that runs the same five operations in both languages: integer
   arithmetic in a loop, array indexing, dictionary/HashMap lookup, string concatenation, and
   object allocation.
2. Use `run_java` with the timing **inside** the JVM, so start-up is excluded.
3. Report the Python:Java ratio for each. They will not be similar — predict the order before you
   run it, then explain the ones you got wrong.
4. Now find where the ratio collapses: rewrite the Python versions to push the work into C
   (`sum()`, `bytes.join`, a comprehension) and re-measure. The gap you can close this way is
   the practical meaning of "write Python that calls C".

***
# Part 5 - Practice

No datasets in this series — the exercises are the practice. Ordered by difficulty.

| # | Exercise | Skill it forces | Difficulty |
|---|---|---|---|
| 1 | Classify by inspection | Reading code for its dominant term | ★☆☆☆☆ |
| 2 | Confirm by measurement | The doubling experiment | ★★☆☆☆ |
| 3 | Find your own crossover | Constants are real | ★★☆☆☆ |
| 4 | Break a benchmark on purpose | Input distribution | ★★★☆☆ |
| 5 | Amortised from scratch | The growth factor | ★★★☆☆ |
| 6 | Solve four recurrences | Master theorem | ★★★☆☆ |
| 7 | The quadratic accident hunt | Recognising it in real code | ★★★★☆ |
| 8 | Adversarial input | Worst cases as a threat model | ★★★★☆ |

***

### 1. Classify by inspection

Give the tight $\Theta$ bound for each, in time and space, then check with `measure_growth`.

```python
def a(xs):                              def b(xs):
    return xs[0] if xs else None            return sorted(set(xs))

def c(xs):                              def d(n):
    out = []                                if n <= 1: return n
    for x in xs:                            return d(n-1) + d(n-2)
        if x in out:
            out.append(x)               def e(xs):
    return out                              return [x for x in xs if x in set(xs)]
```

**The trap:** `c` and `e` look similar and are not. One builds a set once; the other does a linear
scan inside a loop. Say which is which *before* measuring, then measure.

***

### 2. Confirm five claims by measurement

For each, predict the ratio, then run the doubling experiment: `list.append`, `list.insert(0, x)`,
`x in list`, `x in set`, `"".join(parts)` versus `s += part` in a loop.

**A good result:** every prediction confirmed, and you can explain the two that surprised you.
**The trap:** at least one needs sizes far larger than you first choose. §3.2 says why.

***

### 3. Find the crossover on your machine

§1.6 found insertion sort overtaking merge sort at $n = 100$ here.

1. Reproduce it. Do you get the same number? You should not expect to exactly.
2. Now tune it: add a cutoff to merge sort so it calls insertion sort below size $k$. Sweep $k$
   and find the best value. Compare against the cutoff CPython's own sort uses.
3. Explain why the tuned hybrid beats both pure algorithms at every size.

***

### 4. Break a benchmark on purpose

Write a benchmark that "proves" binary search is $O(n)$, and one that "proves" a linear scan is
$O(1)$. Both are achievable without touching either algorithm.

**A good result:** two convincing, wrong benchmarks and a written explanation of the trick in
each. **The point:** you will recognise these in someone else's numbers afterwards, including
your own.

***

### 5. Amortised analysis from scratch

Implement a dynamic array over a fixed-size buffer, with `append`, `get`, `set` and `resize`.

1. Instrument it to count element copies. Confirm $n$ appends cost $< 2n$ copies with doubling.
2. Sweep the growth factor: 1.1, 1.25, 1.5, 2, 4. Plot total copies and peak memory against the
   factor. Where does the time/space trade-off sit, and where would you set it?
3. Implement `pop`, shrinking when the array is a quarter full. Why a quarter and not a half?
   Construct the sequence that makes the half rule $O(n)$ per operation.

***

### 6. Solve four recurrences

$T(n) = 3T(n/2) + n$ · $T(n) = 2T(n/2) + n\log n$ · $T(n) = T(n-1) + n$ ·
$T(n) = T(n/3) + T(2n/3) + n$

Give the closed form, state which Master case applies **or why the theorem does not**, then
verify each by counting with §1.7's `work_per_level`.

**The trap:** two of these are outside the Master theorem. Knowing which is the exercise.

***

### 7. The quadratic accident hunt

Q11 lists four ways $O(n^2)$ appears in code nobody meant to write quadratically.

1. Write a realistic 30-line function containing one of them, well hidden.
2. Measure it, confirm the ratio is ~4, then find and fix the line.
3. Do this for all four patterns.
4. Then go and look for one in code you have actually written. There is usually one.

***

### 8. Adversarial input

§3.3 claims a hash table's $O(n)$ worst case is a threat model rather than a curiosity.

1. Construct a set of Python strings that all land in the same `dict` bucket. (Start from
   `hash()` and `PYTHONHASHSEED=0`; the seed randomisation is the defence you are bypassing.)
2. Measure lookup cost against a set of ordinary keys of the same size.
3. Do the same for quicksort: build the input that forces $\Theta(n^2)$ against a fixed
   first-element pivot, then show a randomised pivot defeats your construction.
4. Write the paragraph you would put in a code review when a service uses user-supplied strings
   as dictionary keys.

***
# Part 6 - Reading

## Start here

**1. *Introduction to Algorithms* (CLRS), Cormen, Leiserson, Rivest & Stein — chapters 2–4 and 17.**
> The standard reference, and these four chapters are the whole of this notebook done properly.
> Ch. 3 is the asymptotic notation, ch. 4 is recurrences and the Master theorem with its proof,
> and **ch. 17 is amortised analysis** — the aggregate, accounting and potential methods that
> §1.5 uses. If you read one thing after this notebook, read ch. 17.

**2. *Algorithms*, Sedgewick & Wayne — section 1.4, "Analysis of Algorithms".**
> The best treatment of the **empirical** side, and the source of the doubling-ratio technique
> §1.4 is built on. Where CLRS proves, Sedgewick measures. The two together are the complete
> picture, and this notebook is deliberately closer to Sedgewick.

**3. *Programming Pearls*, Jon Bentley — columns 6–9.**
> On why the constant factor is not beneath your attention. Bentley's account of taking an
> algorithm through successive speedups, and of when to stop, is the corrective to treating
> Big-O as the only thing that matters. Short, and still the best writing on the subject.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1 — asymptotic notation | **CLRS** ch. 3. **Knuth**, *Big Omicron and big Omega and big Theta*, SIGACT News **1976** — where the $\Omega$/$\Theta$ conventions were settled | 🔍 |
| 1.2 — the cost model and its limits | **Hennessy & Patterson**, *Computer Architecture: A Quantitative Approach*, ch. 2 on memory hierarchy | 🔍 |
| 1.2 — what the numbers are | **Norvig**, *Teach Yourself Programming in Ten Years* — the "latency numbers" table — [norvig.com/21-days.html](https://norvig.com/21-days.html) | ✅ |
| 1.4 — the doubling experiment | **Sedgewick & Wayne**, *Algorithms* §1.4 | 🔍 |
| 1.5 — **amortised analysis** | **CLRS** ch. 17. **Tarjan**, *Amortized computational complexity*, SIAM J. Alg. Disc. Meth. **1985** — the paper that named the potential method | 🔍 |
| 1.6 — hybrid sorts in practice | **Peters**, the CPython `listsort.txt` design note — [github.com/python/cpython](https://github.com/python/cpython/blob/main/Objects/listsort.txt) | ✅ |
| 1.7 — the Master theorem | **CLRS** ch. 4. **Bentley, Haken & Saxe**, *A general method for solving divide-and-conquer recurrences*, SIGACT News **1980** | 🔍 |
| 1.8 — recursion and stack limits | **CPython** `sys.setrecursionlimit` docs; the JVM `-Xss` option | ✅ |
| 3.2 — measurement pitfalls | **Georges, Buytaert & Eeckhout**, *Statistically rigorous Java performance evaluation*, OOPSLA **2007** — why one timing is not a measurement | 🔍 |
| 3.3 — worst cases as threat models | **Crosby & Wallach**, *Denial of Service via Algorithmic Complexity Attacks*, USENIX Security **2003** — the paper that made hash-flooding a real concern and put randomised hashing into every runtime | 🔍 |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**CLRS chapter 17, on amortised analysis.** Asymptotic notation you will absorb by using it, and
the Master theorem is a table you can look up. Amortised analysis is the one idea here that
genuinely does not click without sitting with it, and it is the one that keeps reappearing —
dynamic arrays (NB-01), the two-stack queue (NB-05), Union-Find (NB-11). The potential method in
particular looks like a trick until you have used it twice, and then it looks like the obvious
way to think about a sequence of operations.

Read it alongside §1.5, which measures the growth factor the whole argument depends on, and
Challenge 2, which turns the accounting method into an assertion that either holds or does not.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| "Random" data gives suspiciously clean results | RNG re-seeded per element (§3.1) | Seed once, outside the comprehension |
| An $O(n^2)$ algorithm measures as $O(n)$ | Best-case input — sorted, or all-equal (§1.6) | State and vary the input distribution |
| Everything looks linear | Sizes too small; noise dominates (§3.2) | Runs of tens of ms; range of 100× or more |
| The fitted class changes between runs | Range too narrow; noise beats the $\log n$ term (§3.2) | Widen the range, or count instead of timing |
| Timings jump around between runs | Mean over noisy runs | Take the **minimum**; exclude setup |
| First measurement is much slower | JIT warm-up, cold cache, import cost | Discard warm-up runs; time inside the JVM |
| Recursive code dies on large input | Stack depth, not heap (§1.8) | Explicit stack; do not just raise the limit |
| `sys.getsizeof` reports a suspiciously small number | It excludes referenced objects | Measure the buffer, or use `tracemalloc` |
| Java result differs from Python on the same input | 32-bit `int` overflow (§2.3) | `long`, or `Math.addExact`; cross-check |
| Java build fails on a warning | `-Xlint:all -Werror`, deliberately (§2.1) | Fix the warning; that is the point |
| Big-O improved, wall clock did not | Constant factor, or not the bottleneck (Q12) | Profile before optimising |
| An adversary controls your input | Worst case is a threat model (§3.3) | Randomised pivots, randomised hashing |

## Checklist for a complexity claim

- [ ] Which case — best, average or worst — and does the notation match ($O$ vs $\Theta$)?
- [ ] Is the **input distribution** stated, and is it the one that will occur?
- [ ] Was the generator checked — seeded once, actually varied (§3.1)?
- [ ] Are the sizes large enough that each run takes tens of milliseconds (§3.2)?
- [ ] Is the range wide enough to separate the candidates, and does the verdict **survive a
      second run** (§3.2)?
- [ ] Is the **minimum** of several runs reported, with setup excluded (§1.4)?
- [ ] Does the measured ratio match what the claim predicts — ~2, ~4 (§1.4)?
- [ ] Could this be **counted** instead of timed (§1.7)?
- [ ] Is the runner-up class reported, and is it far enough behind to matter (§2.5)?
- [ ] Is **space** analysed too, including the recursion stack (§1.8)?
- [ ] If the claim is amortised, is the growth **geometric** (§1.5)?
- [ ] Does an adversary choose the input (§3.3)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `arrays_zero_to_hero.ipynb` | §1.5's amortised argument applied properly, plus §1.2's cache effects measured where they are largest |
| `hashing_zero_to_hero.ipynb` | §3.3's adversarial input, constructed and measured |
| `sorting_comparison_zero_to_hero.ipynb` | §1.6's crossover turned into the design of every real sort |
| `recursion_zero_to_hero.ipynb` | §1.7's recurrences and §1.8's stack limits, in depth |

See [`README.md`](README.md) for the full roster and reading order, and
[`plan.qmd`](DSA_ZERO_TO_HERO_PLAN.md) for how each notebook is built and verified.